<a href="https://colab.research.google.com/github/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Readme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Setting up Colab environment...")

    # 1. Clone the repository
    repo_url = "https://github.com/doomguy0991/dls_c.git"
    # Only clone if the directory doesn't exist yet
    if not os.path.exists("/content/dls_c"):
        !git clone $repo_url /content/dls_c

    # 2. Change working directory to the notebook's location so relative paths for libraries work imports
    %cd "/content/dls_c/C4 - Convolutional Neural Networks/Notes"

    print("Setup complete. You can now run the rest of the notebook.")
else:
    print("Running locally or out of Colab. No setup needed.")


# Convolutional Neural Networks

This is the fourth course of the deep learning specialization at [Coursera](https://www.coursera.org/specializations/deep-learning) which is moderated by [DeepLearning.ai](http://deeplearning.ai/). The course is taught by Andrew Ng.

## Table of contents

* [Convolutional Neural Networks](#convolutional-neural-networks)
   * [Table of contents](#table-of-contents)
   * [Course summary](#course-summary)
   * [Foundations of CNNs](#foundations-of-cnns)
      * [Computer vision](#computer-vision)
      * [Edge detection example](#edge-detection-example)
      * [Padding](#padding)
      * [Strided convolution](#strided-convolution)
      * [Convolutions over volumes](#convolutions-over-volumes)
      * [One Layer of a Convolutional Network](#one-layer-of-a-convolutional-network)
      * [A simple convolution network example](#a-simple-convolution-network-example)
      * [Pooling layers](#pooling-layers)
      * [Convolutional neural network example](#convolutional-neural-network-example)
      * [Why convolutions?](#why-convolutions)
   * [Deep convolutional models: case studies](#deep-convolutional-models-case-studies)
      * [Why look at case studies?](#why-look-at-case-studies)
      * [Classic networks](#classic-networks)
      * [Residual Networks (ResNets)](#residual-networks-resnets)
      * [Why ResNets work](#why-resnets-work)
      * [Network in Network and 1×1 convolutions](#network-in-network-and-1-X-1-convolutions)
      * [Inception network motivation](#inception-network-motivation)
      * [Inception network (GoogleNet)](#inception-network-googlenet)
      * [Using Open-Source Implementation](#using-open-source-implementation)
      * [Transfer Learning](#transfer-learning)
      * [Data Augmentation](#data-augmentation)
      * [State of Computer Vision](#state-of-computer-vision)
   * [Object detection](#object-detection)
      * [Object Localization](#object-localization)
      * [Landmark Detection](#landmark-detection)
      * [Object Detection](#object-detection-1)
      * [Convolutional Implementation of Sliding Windows](#convolutional-implementation-of-sliding-windows)
      * [Bounding Box Predictions](#bounding-box-predictions)
      * [Intersection Over Union](#intersection-over-union)
      * [Non-max Suppression](#non-max-suppression)
      * [Anchor Boxes](#anchor-boxes)
      * [YOLO Algorithm](#yolo-algorithm)
      * [Region Proposals (R-CNN)](#region-proposals-r-cnn)
   * [Special applications: Face recognition &amp; Neural style transfer](#special-applications-face-recognition--neural-style-transfer)
      * [Face Recognition](#face-recognition)
         * [What is face recognition?](#what-is-face-recognition)
         * [One Shot Learning](#one-shot-learning)
         * [Siamese Network](#siamese-network)
         * [Triplet Loss](#triplet-loss)
         * [Face Verification and Binary Classification](#face-verification-and-binary-classification)
      * [Neural Style Transfer](#neural-style-transfer)
         * [What is neural style transfer?](#what-is-neural-style-transfer)
         * [What are deep ConvNets learning?](#what-are-deep-convnets-learning)
         * [Cost Function](#cost-function)
         * [Content Cost Function](#content-cost-function)
         * [Style Cost Function](#style-cost-function)
         * [1D and 3D Generalizations](#1d-and-3d-generalizations)
   * [Extras](#extras)
      * [Keras](#keras)

## Course summary

Here is the course summary as given on the course [link](https://www.coursera.org/learn/convolutional-neural-networks):

> This course will teach you how to build convolutional neural networks and apply it to image data. Thanks to deep learning, computer vision is working far better than just two years ago, and this is enabling numerous exciting applications ranging from safe autonomous driving, to accurate face recognition, to automatic reading of radiology images.
>
> You will:
> - Understand how to build a convolutional neural network, including recent variations such as residual networks.
> - Know how to apply convolutional networks to visual detection and recognition tasks.
> - Know to use neural style transfer to generate art.
> - Be able to apply these algorithms to a variety of image, video, and other 2D or 3D data.
>
> This is the fourth course of the Deep Learning Specialization.

### Computer vision

Computer Vision (CV) is a rapidly advancing field of deep learning that enables machines to "see" and interpret visual data. It powers everything from self-driving cars (identifying pedestrians) and Face ID to **Neural Style Transfer** (applying an artist's style to a photo).

#### 1. The "Large Image" Challenge
In traditional fully connected networks, large images create a **parameter explosion**:
*   **64x64 pixel image:** Small, only ~12,288 input features.
*   **1000x1000 pixel image:** 1 million pixels $\times$ 3 color channels (RGB) = **3 million input features**.
*   If the first hidden layer has only 1,000 units, the weight matrix would have **3 billion parameters**. This is computationally expensive and leads to extreme **overfitting**.

**Solution:** Convolutional Neural Networks (CNNs) use the **Convolution** operation to reduce parameters and detect patterns efficiently.

<br>



### The Convolution Operation: Edge Detection



Convolution is the fundamental building block of CNNs. It allows the network to identify low-level features like edges before building up to complex objects.

#### 1. Mechanics of Convolution
To perform a convolution, you slide a small matrix called a **Filter** (or **Kernel**) over an input image and calculate the sum of element-wise products.

*   **Input Image:** Size $n \times n$
*   **Filter:** Size $f \times f$
*   **Output (Feature Map):** Size $(n - f + 1) \times (n - f + 1)$

**Example:** A $6 \times 6$ image convolved with a $3 \times 3$ filter results in a $4 \times 4$ output.



<br>

#### 2. Vertical vs. Horizontal Edge Detection
Filters are designed to react to specific changes in pixel intensity (brightness).

*   **Vertical Edge Filter:** Detects transitions from light to dark (or vice versa) horizontally.
    \begin{bmatrix} 1 & 0 & -1 \\ 1 & 0 & -1 \\ 1 & 0 & -1 \end{bmatrix}
*   **Horizontal Edge Filter:** Detects transitions vertically.
     \begin{bmatrix} 1 & 1 & 1 \\ 0 & 0 & 0 \\ -1 & -1 & -1 \end{bmatrix}
  - An example of convolution operation to detect vertical edges:
    - ![](https://github.com/doomguy0991/dls_c/blob/main/C4%20-%20Convolutional%20Neural%20Networks/Notes/Images/01.png?raw=1)

#### 3. Specialized Filters
Researchers have historically designed specific filters to improve accuracy:
*   **Sobel Filter:** Weights the central row/column more heavily to increase robustness to noise.
*   **Scharr Filter:** An even more aggressive weighting for detecting very sharp edges.

### Padding in CNNs



In deep neural networks, applying multiple convolutions causes the image to shrink significantly and results in the loss of information at the edges. **Padding** is the technique of adding extra borders of pixels (usually zeros) around the input image to solve these issues.



<br>

#### 1. Why do we need Padding?
Without padding, two major problems occur:
1.  **Shrinkage:** If you have a 100-layer network and each layer shrinks the image by a few pixels, you will eventually run out of pixels before reaching the end of the network.
2.  **Edge Information Loss:** Pixels in the center of an image are overlapped by the filter many times, but corner/edge pixels are only "touched" once. Padding ensures edge pixels contribute more to the output.

<br>

#### 2. The Math of Padding
If an $n \times n$ image is convolved with an $f \times f$ filter and padding $p$, the output dimension is:
$$(n + 2p - f + 1) \times (n + 2p - f + 1)$$

*   **$n$**: Input size
*   **$p$**: Padding amount (number of pixels added to each side)
*   **$f$**: Filter size

<br>

#### 3. Types of Convolution
There are two standard choices for padding in deep learning:

##### A. Valid Convolution
*   **Definition:** No padding ($p = 0$).
*   **Effect:** The image shrinks every time.
*   **Output Size:** $(n - f + 1) \times (n - f + 1)$

##### B. Same Convolution
*   **Definition:** Padding is added so that the **output size is exactly the same as the input size**.
*   **Formula to find $p$:** To make the output $n$, we solve the math to get:
    $$p = \frac{f - 1}{2}$$

<br>

#### 4. Why are Filters ($f$) usually Odd?
You will rarely see $2 \times 2$ or $4 \times 4$ filters in research. By convention, $f$ is almost always odd (e.g., $3 \times 3$, $5 \times 5$) for two reasons:
1.  **Symmetric Padding:** If $f$ is odd, the "Same" convolution padding $p$ is an integer. If $f$ were even, you would need asymmetric padding (e.g., more on the left than the right).
2.  **Central Pixel:** Odd-sized filters have a specific **center pixel**, which makes it easier to track the position and orientation of the filter relative to the image.

#### For Computer vision
In your future Computer Vision work, you will mostly use **"Same"** convolutions. This allows you to build very deep architectures (like ResNet or VGG) without worrying about your spatial dimensions disappearing before the final layer.

Do you see how the choice of an odd filter size like $3 \times 3$ makes the padding math much cleaner?

### Strided Convolutions



Strided convolution is a variation of the convolution operation where the filter "jumps" over a specified number of pixels instead of sliding one by one. This is primarily used to reduce the spatial dimensions (width and height) of the image.

<br>

#### 1. The Stride ($s$)
The **Stride** ($s$) is the step size.
*   **Stride = 1:** The filter moves one pixel at a time (standard).
*   **Stride = 2:** The filter skips one pixel, jumping to the second position.

*   **Why use Stride > 1?** It's an alternative to "Pooling" layers. It allows the network to reduce the size of the feature maps, which saves memory and computation while increasing the "Receptive Field" (allowing a single pixel in the next layer to see a larger portion of the original image).
*   **Usage Tip:** Most modern architectures use **Stride = 1** for standard layers and **Stride = 2** specifically when they want to reduce the image dimensions.

#### 2. Output Dimension Formula
If an $n \times n$ image is convolved with an $f \times f$ filter using padding $p$ and stride $s$, the output size is:

$$\text{Output Size} = \left\lfloor \frac{n + 2p - f}{s} + 1 \right\rfloor$$

*   **The Floor Operation ($\lfloor \rfloor$):** We round down. If the filter "hangs over" the edge of the image at its final step, we simply **ignore** that step. The filter must be fully contained within the image (or padded image) boundaries to produce an output.

> **Mathematical Simulation:**
> Let $n=7$ (input), $f=3$ (filter), $p=0$ (no padding), and $s=2$ (stride).
> *   Calculation: $(7 + 0 - 3) / 2 + 1 = \mathbf{3}$
> *   The output is a $3 \times 3$ matrix.

<br>

#### 3. Cross-Correlation vs. Convolution
There is a technical difference between how mathematicians and deep learning researchers define "convolution":

*   **Math/Signal Processing:** You must **flip** the filter horizontally and vertically before sliding it. This provides a property called *associativity*.
*   **Deep Learning:** We skip the flip and just perform the element-wise product. Technically, this is called **Cross-correlation**, but we call it "Convolution" by convention.
*   **Why skip the flip?** In deep learning, the filter values are **learned weights**. If the network needs a flipped filter, it will just learn the flipped values directly. Skipping the flip simplifies the code and works just as well.

<br>


### Convolutions Over Volumes



When working with color images (RGB) or layers within a deep network, convolutions happen in three dimensions ($Height \times Width \times Channels$).

<br>

#### 1. The Channel Rule
The most important rule in 3D convolutions is that the **number of channels in the filter must match the number of channels in the input**.

*   **Input Image:** $6 \times 6 \times \mathbf{3}$ (RGB)
*   **Filter:** $3 \times 3 \times \mathbf{3}$
*   **Channels (Depth):** This 3 represents the Red, Green, and Blue layers.

<br>

#### 2. How the Math Works
Instead of 9 multiplications (for $3 \times 3$), the computer now performs **27 multiplications** ($3 \times 3 \times 3$) at each position and sums them into a **single number**.
*   The filter slides through the height and width, but it covers the entire depth of the input at once.
*   **Crucial Point:** Even though the input and filter are 3D, the output of **one** filter is a **2D matrix** (e.g., $4 \times 4 \times 1$).

<br>

#### 3. Multiple Filters (Creating Volume)
In a real CNN, we don't just want to detect one feature (like vertical edges). We want to detect many (horizontal edges, colors, textures, etc.).

*   If you use **10 different filters**, you get **10 different $4 \times 4$ output maps**.
*   We stack these maps together to create a new 3D volume: $4 \times 4 \times 10$.



#### 4. Dimension Summary
If your input is $n \times n \times n_c$ and you convolve it with $n_f$ filters of size $f \times f \times n_c$:

$$\text{Output Shape} = (n - f + 1) \times (n - f + 1) \times n_f$$

*   **$n_c$**: Number of input channels (e.g., 3 for RGB).
*   **$n_f$**: Number of filters (this becomes the "channels" for the next layer).

<br>

### Anatomy of a Single CNN Layer


A single layer in a Convolutional Neural Network (CNN) performs a transformation similar to a standard fully connected layer ($z = wa + b$), but it uses the **convolution** operation to maintain spatial structure and drastically reduce the number of parameters.

<br>

#### 1. The Forward Propagation Steps
To go from the input activation $a^{[l-1]}$ to the next layer's activation $a^{[l]}$, the network performs three specific steps:

1.  **Convolution:** The input volume is convolved with $n_c^{[l]}$ filters. Filter == W[L], Input == a[L-1].

2.  **Add Bias:** A single real number (bias $b$) is added to each element of the filter's output (using Python broadcasting).
3.  **Activation:** A non-linear function (typically **ReLU**) is applied to the result.

**The Logic:**
$$a^{[l]} = g(\text{Conv}(a^{[l-1]}, W^{[l]}) + b^{[l]})$$



<br>

#### 2. Layer Notation Summary
If layer $l$ is a convolutional layer, the following notations apply:

| Property | Symbol / Formula |
| :--- | :--- |
| **Filter Size** | $f^{[l]}$ |
| **Padding / Stride** | $p^{[l]}$ / $s^{[l]}$ |
| **Input Dimensions ( $a^{[0]}$ )** | $n_H^{[l-1]} \times n_W^{[l-1]} \times n_c^{[l-1]}  $|
| **Output Dimensions (activation $a^{[l]}$ )** | $n_H^{[l]} \times n_W^{[l]} \times n_c^{[l]}$ |
| **Output Dimensionsfor m examples (activation $A^{[l]}$ )** | $n_H^{[l]} \times n_W^{[l]} \times n_c^{[l]}$ |
| **Output Height/Width** | $n_{H/W}^{[l]} = \lfloor \frac{n_{H/W}^{[l-1]} + 2p^{[l]} - f^{[l]}}{s^{[l]}} + 1 \rfloor$ |
| **Filter Dimensions** | $f^{[l]} \times f^{[l]} \times n_c^{[l-1]}$ |
| **#No of filters** | $n_c^{[l]}$ |
| **Weight Tensor ($W^{[l]}$)** | $f^{[l]} \times f^{[l]} \times n_c^{[l-1]} \times n_c^{[l]}$ |
| **Bias ($b^{[l]}$)** | $1 \times 1 \times 1 \times n_c^{[l]}$ |

<br>

### A simple convolution network example


This example demonstrates how spatial dimensions are reduced using different strides and filter sizes.

* **Input Image ($a^{[0]}$):** $39 \times 39 \times 3$
    * $n_H = 39, n_W = 39, n_c = 3$ (RGB)

* **Layer 1 (Convolutional):**
    * **Parameters:** $f=3, s=1, p=0$, with **10 filters**.
    * **Output ($a^{[1]}$):** $37 \times 37 \times 10$
    * *Math:* $\frac{39 + 0 - 3}{1} + 1 = 37$

* **Layer 2 (Convolutional):**
    * **Parameters:** $f=5, s=2, p=0$, with **20 filters**.
    * **Output ($a^{[2]}$):** $17 \times 17 \times 20$
    * *Math:* $\lfloor \frac{37 + 0 - 5}{2} + 1 \rfloor = \lfloor 16 + 1 \rfloor = 17$
    * **Insight:** Shrinking happens much faster here because the **stride ($s=2$)** effectively halves the output size.

* **Layer 3 (Convolutional):**
    * **Parameters:** $f=5, s=2, p=0$, with **40 filters**.
    * **Output ($a^{[3]}$):** $7 \times 7 \times 40$
    * *Math:* $\lfloor \frac{17 + 0 - 5}{2} + 1 \rfloor = \lfloor 6 + 1 \rfloor = 7$

* **Layer 4 (Fully Connected + Softmax):**
    * The 3D volume from Layer 3 ($7 \times 7 \times 40$) is **flattened** into a 1D vector.
    * **Input to FC:** $7 \times 7 \times 40 = \mathbf{1,960}$ **units**.
    * This vector is then fed into a Softmax layer for multi-class classification.

#### The Three Pillars: Layer Types
Most CNN architectures are composed of these three fundamental layer types:

| Layer Type | Abbreviation | Purpose |
| :--- | :--- | :--- |
| **Convolutional** | **#Conv** | Feature extraction through learned filters. |
| **Pooling** | **#Pool** | Downsampling to reduce spatial size and parameter count (e.g., Max Pool). |
| **Fully Connected** | **#FC** | Standard neural network layer used at the end for classification/regression. |



### Pooling Layers



Pooling layers are used in ConvNets primarily to **reduce the spatial size** (downsampling) of the representation. This speeds up computation, reduces the number of parameters in subsequent layers (preventing overfitting), and makes feature detection more robust to small distortions or shifts in the image.

<br>

#### Max Pooling
This is the most common type of pooling. For every region covered by the filter, it selects the **maximum value**.

* **Logic:** If we treat the numbers as "feature activations," a high number means a feature (like a cat's ear or a vertical edge) was detected. Max pooling says: "If this feature exists *anywhere* in this small region, preserve it."
* **Hyperparameters:** Usually $f=2, s=2$. This effectively halves the height and width of the image.



<br>

#### Average Pooling
Instead of taking the maximum, this layer calculates the **average value** of all pixels in the filter region.

* **Usage:** It is much less common than Max Pooling in early/middle layers.
* **Exception:** Often used at the very end of deep architectures (Global Average Pooling) to collapse a volume like $7 \times 7 \times 1000$ into a $1 \times 1 \times 1000$ vector before the final classification.

<br>

#### Crucial Understandings

* **No Learnable Parameters:** Unlike Convolutional layers, pooling has **nothing for Gradient Descent to learn**. Once you choose the filter size ($f$) and stride ($s$), the operation is a fixed mathematical function.
* **Channel Independence:** Pooling is performed on each channel **independently**. If the input is $n_H \times n_W \times n_C$, the output will still have $n_C$ channels. The depth never changes during a pooling operation.
* **Padding:** It is very rare to use padding in pooling layers. By default, $p = 0$.

<br>

#### Dimensions & Formula
The output size follows the same formula as the convolution layer:

$$n_{H/W}^{[l]} = \left\lfloor \frac{n_{H/W}^{[l-1]} + 2p - f}{s} + 1 \right\rfloor$$

**Common Hyperparameter Combinations:**
* $f=2, s=2$ (Most common: reduces size by 50%)
* $f=3, s=2$ (Overlapping pooling)

### Why Convolutions?



Compared to standard Fully Connected (FC) layers, Convolutional layers are far more efficient for image data. This efficiency allows us to build deep networks for high-resolution images without the "parameter explosion" that would otherwise occur.

<br>

#### 1. Two Main Advantages

##### A. Parameter Sharing
A feature detector (like a vertical edge filter) that is useful in the top-left of an image is likely useful in the bottom-right as well.
* **How it works:** Instead of learning different weights for every pixel location, we learn **one filter** and slide it across the entire image.
* **Result:** All positions in the input image share the same parameters. This drastically reduces the number of values the model needs to learn.



##### B. Sparsity of Connections
In a convolutional layer, each output value depends only on a small number of inputs (the size of the filter), not the entire image.
* **Example:** In a $3 \times 3$ convolution, a single output unit is "connected" to only 9 input pixels.
* **Result:** This localized connection makes the network **Translation Invariant**. Because the model looks at local patterns, a "cat" shifted by 10 pixels is still recognized as a cat because the same filter will eventually slide over it and "fire."



<br>

#### 2. Parameter Comparison: FC vs. Conv
Consider a small $32 \times 32 \times 3$ image (3,072 features) mapped to a $28 \times 28 \times 6$ output (4,704 features).

* **Fully Connected Layer:** Every input connects to every output.
    * Parameters $\approx 3,072 \times 4,704 \approx \mathbf{14.4\text{ million}}$
* **Convolutional Layer ($5 \times 5$ filter, 6 filters):**
    * Parameters $= 6 \times (5 \times 5 \times 3 + 1) = \mathbf{456}$

The reduction from 14 million to 456 parameters is staggering. This efficiency prevents **overfitting** and makes training on smaller datasets feasible.

<br>

#### 3. Putting It All Together: The Training Flow
To build an end-to-end system (like a cat detector), we stack these blocks together:

1.  **Architecture:** Input $\rightarrow$ [Conv $\rightarrow$ Pool] layers $\rightarrow$ [FC] layers $\rightarrow$ **Softmax Output**.
2.  **Cost Function ($J$):** We define the cost as the average loss ($L$) across all $m$ training examples:

    $$J = \frac{1}{m} \sum_{i=1}^{m} L(\hat{y}^{(i)}, y^{(i)})$$
3.  **Optimization:** We use algorithms like **Adam**, **Momentum**, or **RMSProp** to update all weights ($W$) and biases ($b$) to minimize $J$.

<br>

